In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lmfit import Model

from functions.m_statistical_calculations import exponential_decay
from functions.m_statistical_calculations import logistic_decay
# plot mean and std data for different machines

# set machine
set_machine = 'DLRA'  # Options: 'OCEAN', 'DLRA'

# read data
df_mean_std = pd.read_csv(
    f'output/{set_machine}/mean_std_datarecorder_with_all_measures.csv'
)
df_dryness = pd.read_csv(
    f'output/{set_machine}/dryness_data.csv'
)

# axis & hue logic
if set_machine == 'OCEAN':
    x = 't_duration'
    hue = None
elif set_machine == 'DLRA':
    x = 'n_UL'
    hue = 'T_drying'

# create figure
plt.figure(figsize=(10, 6))

# ---- scatter: all raw values (background) ----
sns.scatterplot(
    data=df_dryness,
    x=x,
    y='m_diff',
    hue=hue,
    alpha=0.35,
    legend=True
)

# ---- line: mean ----
sns.lineplot(
    data=df_mean_std,
    x=x,
    y='m_water_mean',
    hue=hue,
    marker='o',
    legend=False  # avoid duplicate legend
)

# ---- shaded std band ----
ax = plt.gca()

if hue:
    hues = df_mean_std[hue].unique()
    colors = {h: ax.lines[i].get_color() for i, h in enumerate(hues)}

    for h in hues:
        grp = df_mean_std[df_mean_std[hue] == h]

        # 1. Fill between ± std
        plt.fill_between(
            grp[x],
            grp['m_water_mean'] - grp['m_water_std'],
            grp['m_water_mean'] + grp['m_water_std'],
            alpha=0.25,
            color=colors[h],
        )

        # 2. Quadratic regplot
        sns.regplot(
            data=df_dryness[df_dryness[hue] == h],
            x=x,
            y='m_diff',
            scatter=False,
            ci=None,
            color=colors[h],
            line_kws={'linestyle': '--', 'linewidth': 2},
            ax=ax,
            order=2
        )

        # 3. exponential fit using lmfit
        x_data = df_dryness[df_dryness[hue] == h][x].values
        y_data = df_dryness[df_dryness[hue] == h]['m_diff'].values

        # Step 1: Rescale x to make fitting stable
        # --------------------------
        x_scaled = x_data / 1000  # scale to ~1–3 instead of 1000–3000

        # --------------------------
        # Step 2: Create lmfit model and parameters
        # --------------------------
        exp_model = Model(exponential_decay)
        logistic_model = Model(logistic_decay)


        # Good initial guesses
        params_exp = exp_model.make_params(
            A=y_data.max() - y_data.min(),  # amplitude
            k=1.0,                          # initial decay rate (scaled units)
            C=y_data.min()                  # offset
        )
        params_logistic = logistic_model.make_params(
            L=y_data.max(),  # maximum value
            k=1.0,           # growth rate
            x0=np.median(x_scaled),   # midpoint
            C =y_data.min()  # offset
        )

        params_exp['k'].set(min=0)  # decay rate must be positive
        params_logistic['k'].set(min=0)  # growth rate must be positive
        params_logistic['L'].set(min=0)  # L must be positive

        # --------------------------
        # Step 3: Fit using all data points (including repeats)
        # --------------------------
        result_exp = exp_model.fit(y_data, params_exp, x=x_scaled)
        result_logistic = logistic_model.fit(y_data, params_logistic, x=x_scaled)
        # --------------------------
        # Step 4: Check fitted parameters
        # --------------------------
        print(result_exp.fit_report())
        print(result_logistic.fit_report())

        # --------------------------
        # Step 5: Plot
        # --------------------------
        x_fit = np.linspace(x_scaled.min(), x_scaled.max(), 500)
        y_fit_exp = result_exp.eval(x=x_fit)
        y_fit_logistic = result_logistic.eval(x=x_fit)

        # 4. Plot fitted curve with rescaled x
        plt.plot(x_fit * 1000, y_fit_exp, 'r-', linewidth=2, label='Exponential fit')
        plt.plot(x_fit * 1000, y_fit_logistic, 'g--', linewidth=2, label='Logistic fit')
else:
    plt.fill_between(
        df_mean_std[x],
        df_mean_std['m_water_mean'] - df_mean_std['m_water_std'],
        df_mean_std['m_water_mean'] + df_mean_std['m_water_std'],
        alpha=0.25
    )
    sns.regplot(
        data=df_dryness,
        x=x,
        y='m_diff',
        scatter=False,
        ci=None,
        line_kws={'linestyle': '--', 'linewidth': 2},
        ax=ax,
        order=2
    )



# ---- labels & styling ----
plt.title(f'Dryness Values with Mean ± Std for {set_machine}')
plt.xlabel('Drying Time (s)' if x == 't_duration' else 'NRotation Speed of UL')
plt.ylabel('Dryness Value')
plt.grid(True)
plt.legend(title='Drying Temperature (T_drying)' if hue else None)

# save & show
plt.savefig(f'output/{set_machine}/mean_std_dryness_plot.png', dpi=300, bbox_inches='tight')
plt.show()


ImportError: cannot import name 'logistic_decay' from 'functions.m_statistical_calculations' (c:\Users\nilst\OneDrive\Dokumente\UNI\Arbeit ETA\drysense-2025_2026\functions\m_statistical_calculations.py)